# 08 — Producto: el execute iguala al 07

No envuelve a mano. Llama `AlgorithmRegistry.execute` (el `run()` de la API)
y lo compara con los brazos `consolida` del `07`.

Control: `00100408` con `consolidate=false` debe partir la carga.
Siguiente: `09_homologar_pct.ipynb`.


In [1]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")

REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("ML_ROOT   =", ML_ROOT)
print("REPO_ROOT =", REPO_ROOT)

ML_ROOT   = /home/edmundo/packing-services/online_policy_ml
REPO_ROOT = /home/edmundo/packing-services


## Comprobaciones

Encoder v1, checkpoints de producción intactos, Paso A sellado y exitoso.

In [2]:
from consolidation import (
    PASO_A_LABEL,
    PRODUCT_ARMS,
    OnlineArm,
    run_paso_c,
    run_via_execute,
)
from paths import HOLDOUT_DIR
from registry import (
    STATUS_EN_CURSO,
    STATUS_VALIDADO,
    load_sealed_report,
    registry_table,
    seal_run,
)
from splits import load_orders

from packing_services.online.features import FEATURE_DIM, FEATURE_VERSION
from packing_services.online.learned.production import (
    CINTA_MODEL_PATH,
    DEFAULT_MODEL_PATH,
    PRODUCT_HOLDOUT_ORDER_IDS,
)
from packing_services.online.params import wants_consolidate

FLOW_ID = "eval_pasoC_producto"
NOTEBOOK = "online_policy_ml/notebooks/08_producto_execute.ipynb"
CONTAINERS = 2

assert FEATURE_VERSION == 1 and FEATURE_DIM == 35
assert wants_consolidate({}, 1) is False
assert wants_consolidate({}, 2) is True

MODELS = {name: REPO_ROOT / name for name in (DEFAULT_MODEL_PATH, CINTA_MODEL_PATH)}
for name, path in MODELS.items():
    assert path.is_file(), f"falta {name}"
MTIMES_ANTES = {name: path.stat().st_mtime for name, path in MODELS.items()}

paso_a = load_sealed_report("eval_pasoA_consolidacion")
assert paso_a["paso_a_exitoso"], "el Paso A no está validado; no se sigue"

ORDERS = load_orders(HOLDOUT_DIR / "5_bed-bpp.json")
ORDER_IDS = sorted(ORDERS)
assert set(ORDER_IDS) == set(PRODUCT_HOLDOUT_ORDER_IDS)

print("encoder v1", FEATURE_DIM, "features")
print("Paso A exitoso:", paso_a["paso_a_exitoso"])
print("pallets A:", paso_a.get("criterios"))
print("brazos de producto:")
for arm in PRODUCT_ARMS:
    print(f"  {arm.label:<14} p={arm.lookahead_p} s={arm.select_s} consolidate={arm.consolidate}")

encoder v1 35 features
Paso A exitoso: True
pallets A: {'ahorra_pallets': True, 'no_deja_mas_items_fuera': True, 'todo_valido': True, 'no_empeora_cubo': True}
brazos de producto:
  greedy p1s1    p=1 s=1 consolidate=None
  rl_p1s1        p=1 s=1 consolidate=None
  mlp_p3s2       p=3 s=2 consolidate=None


## Control en `00100408`

Con `consolidate=false` el heurístico debe partir 14/12. Con el default, 26 en un pallet.

In [3]:
from packing_services.online.learned.production import DEFAULT_LOOKAHEAD_P, DEFAULT_SELECT_S

control_off = OnlineArm("greedy libre", DEFAULT_LOOKAHEAD_P, DEFAULT_SELECT_S, consolidate=False)
control_on = OnlineArm("greedy consolida", DEFAULT_LOOKAHEAD_P, DEFAULT_SELECT_S, consolidate=True)
off = run_via_execute(ORDERS, "00100408", control_off, containers=CONTAINERS)
on = run_via_execute(ORDERS, "00100408", control_on, containers=CONTAINERS)
control_df = pd.DataFrame([off, on])[
    ["arm", "containers_used", "volume_utilization", "items_packed", "spread", "is_valid"]
]
display(control_df)
assert off["containers_used"] == 2, "sin disciplina debería partir la carga"
assert on["containers_used"] == 1, "con disciplina 00100408 cabe en un pallet"
assert on["items_packed"] == 26 and off["items_packed"] == 26
assert off["spread"] == {"EURO_PALLET-1": 14, "EURO_PALLET-2": 12}, "el control debe ser el 14/12 de Fase 1"
assert on["spread"] == {"EURO_PALLET-1": 26}
print("control OK: false = 14/12; true = 26 en un pallet")

,arm,containers_used,volume_utilization,items_packed,spread,is_valid
0,greedy libre,2,0.3232,26,"{'EURO_PALLET-1': 14, 'EURO_PALLET-2': 12}",True
1,greedy consolida,1,0.6464,26,{'EURO_PALLET-1': 26},True


control OK: false = 14/12; true = 26 en un pallet


## Holdout por el execute

Quince corridas (5 pedidos × 3 motores). Tarda unos minutos.

In [4]:
rows = run_paso_c(ORDERS, ORDER_IDS, containers=CONTAINERS, verbose=True)
df = pd.DataFrame(rows)
assert df["is_valid"].all()
assert (df["constraint_violations"] == 0).all()
assert (df["items_unpacked"] == 0).all()
print(len(rows), "corridas validas, 0 items fuera")
display(df.pivot_table(
    index="order_id",
    columns="arm",
    values=["containers_used", "volume_utilization"],
))

00100001 · greedy p1s1    pallets=2 emp=44/44 util=39.3% t=1.25s
00100001 · rl_p1s1        pallets=2 emp=44/44 util=39.3% t=1.17s
00100001 · mlp_p3s2       pallets=2 emp=44/44 util=39.3% t=2.65s
00100002 · greedy p1s1    pallets=2 emp=38/38 util=40.1% t=0.95s
00100002 · rl_p1s1        pallets=2 emp=38/38 util=40.1% t=1.02s
00100002 · mlp_p3s2       pallets=2 emp=38/38 util=40.1% t=1.69s
00100003 · greedy p1s1    pallets=2 emp=34/34 util=43.9% t=0.49s
00100003 · rl_p1s1        pallets=2 emp=34/34 util=43.9% t=0.64s
00100003 · mlp_p3s2       pallets=2 emp=34/34 util=43.9% t=1.22s
00100004 · greedy p1s1    pallets=1 emp=58/58 util=61.4% t=6.81s
00100004 · rl_p1s1        pallets=1 emp=58/58 util=61.4% t=6.38s
00100004 · mlp_p3s2       pallets=1 emp=58/58 util=61.4% t=14.34s
00100408 · greedy p1s1    pallets=1 emp=26/26 util=64.6% t=0.68s
00100408 · rl_p1s1        pallets=1 emp=26/26 util=64.6% t=0.51s
00100408 · mlp_p3s2       pallets=1 emp=26/26 util=64.6% t=1.09s
15 corridas validas, 0 i

containers_used                  volume_utilization                 
arm          greedy p1s1 mlp_p3s2 rl_p1s1        greedy p1s1 mlp_p3s2 rl_p1s1
order_id                                                                     
00100001          2.0000   2.0000  2.0000             0.3925   0.3925  0.3925
00100002          2.0000   2.0000  2.0000             0.4012   0.4012  0.4012
00100003          2.0000   2.0000  2.0000             0.4387   0.4387  0.4387
00100004          1.0000   1.0000  1.0000             0.6142   0.6142  0.6142
00100408          1.0000   1.0000  1.0000             0.6464   0.6464  0.6464

## Contra el Paso A consolidado

Misma etiqueta de motor, mismos pallets, cubo a 1e-4.

In [5]:
prev = pd.DataFrame(paso_a["runs"])
prev_cons = prev[prev["discipline"] == "consolida"].copy()
check = []
for row in rows:
    etiqueta = PASO_A_LABEL[row["pair_key"]]
    base = prev_cons[(prev_cons["order_id"] == row["order_id"]) & (prev_cons["arm"] == etiqueta)]
    assert len(base) == 1, f"no hay brazo A {etiqueta} para {row['order_id']}"
    old = base.iloc[0]
    check.append({
        "order_id": row["order_id"],
        "motor": row["pair_key"],
        "pallets_A": int(old["containers_used"]),
        "pallets_API": int(row["containers_used"]),
        "util_A": float(old["volume_utilization"]),
        "util_API": float(row["volume_utilization"]),
        "items_A": int(old["items_packed"]),
        "items_API": int(row["items_packed"]),
    })
check_df = pd.DataFrame(check)
display(check_df)
reproduce = bool(
    (check_df["pallets_A"] == check_df["pallets_API"]).all()
    and (check_df["items_A"] == check_df["items_API"]).all()
    and (check_df["util_A"] - check_df["util_API"]).abs().max() < 1e-4
)
print("el execute reproduce el Paso A consolidado:", reproduce)
if not reproduce:
    print("ATENCION: no selles; el cableado no coincide con la medicion")

,order_id,motor,pallets_A,pallets_API,util_A,util_API,items_A,items_API
0,00100001,greedy#p1s1,2,2,0.3925,0.3925,44,44
1,00100001,aprendida#p1s1,2,2,0.3925,0.3925,44,44
2,00100001,aprendida#p3s2,2,2,0.3925,0.3925,44,44
3,00100002,greedy#p1s1,2,2,0.4012,0.4012,38,38
4,00100002,aprendida#p1s1,2,2,0.4012,0.4012,38,38
5,00100002,aprendida#p3s2,2,2,0.4012,0.4012,38,38
6,00100003,greedy#p1s1,2,2,0.4387,0.4387,34,34
7,00100003,aprendida#p1s1,2,2,0.4387,0.4387,34,34
8,00100003,aprendida#p3s2,2,2,0.4387,0.4387,34,34
9,00100004,greedy#p1s1,1,1,0.6142,0.6142,58,58


el execute reproduce el Paso A consolidado: True


## Criterios y brecha restante

Tres pedidos siguen en dos pallets: no caben en uno. Eso no es el defecto de la Fase 1.

In [6]:
criterios = {
    "todo_valido": bool(df["is_valid"].all() and (df["items_unpacked"] == 0).all()),
    "reproduce_paso_A": reproduce,
    "control_reparte": off["containers_used"] == 2,
    "control_consolida": on["containers_used"] == 1,
}
paso_c_exitoso = all(criterios.values())
print("Criterios")
for nombre, ok in criterios.items():
    print(f"  {'OK ' if ok else 'NO '} {nombre}")
print()
print("PASO C EXITOSO" if paso_c_exitoso else "PASO C NO CONCLUYENTE")
print()
por_pedido = df.groupby("order_id").agg(
    pallets=("containers_used", "max"),
    util=("volume_utilization", "mean"),
    items=("n_items", "first"),
).reset_index()
display(por_pedido)
print("pedidos que siguen en 2 pallets (no caben en uno):")
print(por_pedido.loc[por_pedido["pallets"] == 2, "order_id"].tolist())

Criterios
  OK  todo_valido
  OK  reproduce_paso_A
  OK  control_reparte
  OK  control_consolida

PASO C EXITOSO



,order_id,pallets,util,items
0,00100001,2,0.3925,44
1,00100002,2,0.4012,38
2,00100003,2,0.4387,34
3,00100004,1,0.6142,58
4,00100408,1,0.6464,26


pedidos que siguen en 2 pallets (no caben en uno):
['00100001', '00100002', '00100003']


## Sellado

Solo `validado` si los cuatro criterios pasaron. Los checkpoints no deben haberse movido.

In [7]:
for name, path in MODELS.items():
    assert path.stat().st_mtime == MTIMES_ANTES[name], f"{name} cambio"

report = {
    "paso": "C",
    "pregunta": "el execute de produccion iguala al Paso A consolidado",
    "containers_available": CONTAINERS,
    "order_ids": ORDER_IDS,
    "encoder": {"feature_version": FEATURE_VERSION, "feature_dim": FEATURE_DIM},
    "checkpoints": {name: str(path) for name, path in MODELS.items()},
    "arms": [
        {
            "label": a.label,
            "family": a.family,
            "lookahead_p": a.lookahead_p,
            "select_s": a.select_s,
            "model_path": a.model_path,
        }
        for a in PRODUCT_ARMS
    ],
    "runs": rows,
    "versus_paso_A": check,
    "control_00100408": {"libre": off, "consolida": on},
    "criterios": criterios,
    "paso_c_exitoso": paso_c_exitoso,
    "nota": "camino AlgorithmRegistry.execute; no reentrena; PPO cerrado",
}
headline = {
    "pallets": int(df["containers_used"].sum()),
    "util": round(float(df["volume_utilization"].mean()), 6),
    "items_fuera": int(df["items_unpacked"].sum()),
    "reproduce_paso_A": reproduce,
    "paso_c_exitoso": paso_c_exitoso,
}
entry = seal_run(
    FLOW_ID,
    title="Paso C: execute de producto vs Paso A consolidado",
    source=NOTEBOOK,
    report=report,
    headline=headline,
    status=STATUS_VALIDADO if paso_c_exitoso else STATUS_EN_CURSO,
    notes="Registry.execute con consolidate default. Sin reentrenamiento.",
)
print("version sellada:", entry["version"], "estado:", entry["status"])
print("carpeta:", entry["path"])
display(pd.DataFrame(registry_table()))

version sellada: 3 estado: validado
carpeta: artifacts/runs/eval_pasoC_producto/v3


,flujo,titulo,versiones,ultima,estado,sellado,fuente
0,eval_fase0_baseline_p3s2,Fase 0 complemento: baseline greedy al mismo p...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
1,eval_fase0_holdout,Fase 0: comparacion de producto offline vs onl...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
2,eval_fase1_multipallet,Fase 1: multi-pallet con dos contenedores iden...,1,v1,validado,2026-09-11,scripts/run_holdout_offline_vs_online.py
3,eval_pasoA_consolidacion,Paso A: disciplina de consolidacion en multi-p...,3,v3,validado,2026-09-15,online_policy_ml/notebooks/07_multipallet_firs...
4,eval_pasoC_producto,Paso C: execute de producto vs Paso A consolidado,3,v3,validado,2026-09-15,online_policy_ml/notebooks/08_producto_execute...
5,ml_fase1_datos_splits,Fase 1: datos BED-BPP y splits,1,v1,validado,2026-09-11,online_policy_ml/notebooks/01_datos_y_splits.i...
6,ml_fase2_etiquetas_p2o,Fase 2: transiciones P2O etiquetadas por el ma...,1,v1,validado,2026-09-11,online_policy_ml/notebooks/02_etiquetas_p2o.ipynb
7,ml_fase3_entrenar_exportar,"Fase 3: entrenar, validar y exportar checkpoints",1,v1,validado,2026-09-11,online_policy_ml/notebooks/03_entrenar_validar...
8,ml_fase4_escalar,Fase 4: escalar a pedidos medianos y revalidar,1,v1,validado,2026-09-11,online_policy_ml/notebooks/04_escalar_revalida...
9,ml_fase5_receding_p3s2,Fase 5: maestro receding-horizon para el regim...,1,v1,validado,2026-09-11,online_policy_ml/notebooks/05_receding_horizon...
